In [0]:
-- ============================================================
-- CARE CASCADE — Hypertension management funnel
-- Four stacked CTEs, each an INNER JOIN to the prior stage's
-- survivors against a new fact table. Measures patient drop-off
-- through the ideal care pathway:diagnosed→followed up→medicated→controlled
-- ============================================================

WITH
-- STAGE 1: patients diagnosed with essential hypertension,
-- taking each patient's FIRST diagnosis date as the cascade start.
diagnosed AS (
  SELECT
    patient_id,
    MIN(onset_date) AS dx_date
  FROM health_lakehouse.gold.fact_conditions
  WHERE condition_code = '59621000' -- Essential hypertension
  GROUP BY patient_id
),

-- STAGE 2: of those, who had ANY encounter within 90 days AFTER diagnosis
-- (a follow-up touchpoint — the chance to act on the diagnosis).
followed_up AS (
  SELECT DISTINCT d.patient_id, d.dx_date
  FROM diagnosed d
  JOIN health_lakehouse.gold.fact_encounters e
    ON e.patient_id = d.patient_id
   AND e.encounter_date >  d.dx_date
   AND e.encounter_date <= d.dx_date + INTERVAL 90 DAYS
),

-- STAGE 3: of those, who were prescribed a BP-lowering medication
-- AFTER diagnosis (matched on drug stems, all variants + combos).
medicated AS (
  SELECT DISTINCT f.patient_id, f.dx_date,
         MIN(m.start_date) AS first_med_date
  FROM followed_up f
  JOIN health_lakehouse.gold.fact_medications m
    ON m.patient_id = f.patient_id
   AND m.start_date >= f.dx_date
   AND LOWER(m.medication_desc) RLIKE
       'lisinopril|hydrochlorothiazide|amlodipine|losartan|metoprolol|furosemide|valsartan|enalapril|atenolol|benazepril|bisoprolol|olmesartan|sacubitril'
  GROUP BY f.patient_id, f.dx_date
),

-- STAGE 4: of those, who later had a systolic BP reading < 140 mmHg
-- (controlled), recorded AFTER their first medication.
controlled AS (
  SELECT DISTINCT mo.patient_id
  FROM medicated mo
  JOIN health_lakehouse.gold.fact_observations o
    ON o.patient_id = mo.patient_id
   AND o.measure_group = 'vital: systolic_bp'
   AND o.observed_ts >= mo.first_med_date
   AND o.value_numeric < 140
)

-- FUNNEL: count survivors at each stage + retention vs. previous.
SELECT stage, patients,
       ROUND(100.0 * patients / FIRST(patients) OVER (ORDER BY stage_order), 1) AS pct_of_diagnosed,
       ROUND(100.0 * patients / LAG(patients) OVER (ORDER BY stage_order), 1)   AS pct_of_previous
FROM (
  SELECT 1 AS stage_order, '1. Diagnosed' AS stage, COUNT(*) AS patients FROM diagnosed
  UNION ALL
  SELECT 2, '2. Followed up (≤90d, guideline target)', COUNT(*) FROM followed_up
  UNION ALL
  SELECT 3, '3. Medicated', COUNT(*) FROM medicated
  UNION ALL
  SELECT 4, '4. Controlled (<140)', COUNT(*) FROM controlled
)
ORDER BY stage_order;

In [0]:
-- A 33.7% follow up rate within 90 days is alarming (probe further)
-- Of diagnosed patients, how many have ANY encounter after dx at all?,
-- and what does the gap to next encounter actually look like?
WITH diagnosed AS (
  SELECT patient_id, MIN(onset_date) AS dx_date
  FROM health_lakehouse.gold.fact_conditions
  WHERE condition_code = '59621000'
  GROUP BY patient_id
),
next_enc AS (
  SELECT d.patient_id, d.dx_date,
         MIN(e.encounter_date) AS next_enc_date
  FROM diagnosed d
  LEFT JOIN health_lakehouse.gold.fact_encounters e
    ON e.patient_id = d.patient_id
   AND e.encounter_date > d.dx_date
  GROUP BY d.patient_id, d.dx_date
)
SELECT
  COUNT(*) AS total_diagnosed,
  SUM(CASE WHEN next_enc_date IS NULL THEN 1 ELSE 0 END) AS no_later_encounter,
  SUM(CASE WHEN DATEDIFF(next_enc_date, dx_date) <= 90 THEN 1 ELSE 0 END) AS within_90d,
  SUM(CASE WHEN DATEDIFF(next_enc_date, dx_date) BETWEEN 91 AND 365 THEN 1 ELSE 0 END) AS within_1yr,
  SUM(CASE WHEN DATEDIFF(next_enc_date, dx_date) > 365 THEN 1 ELSE 0 END) AS over_1yr
FROM next_enc;

In [0]:
-- ============================================================
-- CARE CASCADE — companion: guideline vs. actual follow-up timing
-- The cascade uses a 90-day follow-up window (the guideline-
-- recommended interval for a new hypertension diagnosis).
-- Most patients DO return — but on an annual cadence, not 90 days.
-- This quantifies the gap between recommended and actual practice.
-- ============================================================
WITH diagnosed AS (
  SELECT patient_id, MIN(onset_date) AS dx_date
  FROM health_lakehouse.gold.fact_conditions
  WHERE condition_code = '59621000'
  GROUP BY patient_id
),
next_enc AS (
  SELECT d.patient_id, d.dx_date,
         MIN(e.encounter_date) AS next_enc_date
  FROM diagnosed d
  LEFT JOIN health_lakehouse.gold.fact_encounters e
    ON e.patient_id = d.patient_id AND e.encounter_date > d.dx_date
  GROUP BY d.patient_id, d.dx_date
)
SELECT
  CASE
    WHEN next_enc_date IS NULL THEN '0. No later visit'
    WHEN DATEDIFF(next_enc_date, dx_date) <= 90 THEN '1. Within 90 days (guideline)'
    WHEN DATEDIFF(next_enc_date, dx_date) <= 365 THEN '2. 91-365 days'
    ELSE '3. Over 1 year'
  END AS follow_up_timing,
  COUNT(*) AS patients,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM next_enc
GROUP BY 1
ORDER BY 1;